In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )

Rdair=Co.Rdair()


In [ ]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [ ]:
%%time
nsteps=None
start_date=None
#super_lat_range = [-90.,90.]  #[-85,-30]
super_lat_range = [-80.,-30.]  #[-85,-30]
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False, [2004,7,15,0], 248
#case, process_ncdata  = 'cam77_dyamond1_prod1'    , False
case, process_ncdata  = 'c124_dyamond1_prod2'    , False
#case , process_ncdata = 'xy-rdg-mm-front'    , True
A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon



In [ ]:
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)

zlev_event=15_000. #23_000.
lat_range=  [-50,-40] #[-70,-60] #[-60,-40]
lat_range=  [-60,-40] #[-70,-60] #[-60,-40]
#lat_range=  [35,65] # Northern Summer!!!!!!!!!!
lon_range=[0,360] # [0,60]
exclude_orography=True

fracs=[0.995,0.90,0.50,0.25,0.125,0.0625]

El = euti.make_El(
            A=A, 
            fractions_for_thresholds=fracs,
            zlev_event=zlev_event, 
            lat_range=lat_range,
            lon_range=lon_range,
            exclude_orography= exclude_orography
           )




In [ ]:

ds=El[2].ds
print(int(ds.itime.values.max()) + 1)

ds_drop=ds.drop_vars( ['lat','lon','zlev'] )


event_list=euti.ds_to_event_list( ds_drop )

In [ ]:
# event_list[t] must contain dicts with 't' key added
# (as we discussed earlier when adding timestep to each event)
importlib.reload( euti )

tracks = euti.track_events(
    event_list    = event_list,
    lat           = lat,
    lon           = lon,
    dt_hours      = 3.0,
    max_speed_kmh = 180.0,    # ~Southern Ocean cyclone speed
    min_lifetime  = 1,       # require at least 2 timesteps = 6 hours
)

_ = euti.plot_track_statistics_2(tracks, dt_hours=3.0)

In [ ]:
80_000./3600.

In [ ]:
A.rho_epwp.shape

In [ ]:
lfes=tracks[0]['lifetime']

tracks[0].keys()

In [ ]:
lifetimes  = np.array([tr['lifetime']   for tr in tracks])
speeds     = np.array([tr['mean_speed'] for tr in tracks])
maxlife    = np.max(lifetimes)+1
ntracks    = len(lifetimes)


shape = (ntracks, maxlife)

ixs   = np.full(shape, -1, dtype=int)
iys   = np.full(shape, -1, dtype=int)
times = np.full(shape, -1, dtype=int)

itrk=0
for tr in tracks:
    length=len(tr['times'])
    times[itrk,0:length] = tr['times']
    itrk=itrk+1

#vel_y_all  = np.concatenate([tr['vel_y_kmh'][1:] for tr in tracks])
#vel_x_all  = np.concatenate([tr['vel_x_kmh'][1:] for tr in tracks])
#ixs        = np.concatenate([tr['ix'] for tr in tracks])
#iys        = np.concatenate([tr['iy'] for tr in tracks])
#times      = np.concatenate([tr['times'] for tr in tracks])


In [ ]:
print( times[400,:] )

In [ ]:
plt.plot(lifetimes,'o')
plt.xlim(790,800)

In [ ]:
ixs.shape

In [ ]:
importlib.reload(euti)
trx=euti.trackarrays( tracks, lon=A.lon,lat=A.lat )

In [ ]:
itx=791
plt.scatter( trx.lons[itx,:]  , trx.lats[itx,:] )

print( trx.times[itx,:] )

In [ ]:
z0=np.argmin( np.abs( zlev-0.))
z0p5=np.argmin( np.abs( zlev-500))
z1=np.argmin( np.abs( zlev-1000.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))

tim=144
oo=np.where( trx.times == tim )


A.lat.shape
plt.contourf(lon,lat, np.log(A.epwp[tim,z15,:,:]+1e-6)  )
plt.contour(lon,lat, A.htopo ,levels=[1.e-12,0.1,1,10,100,1000])

for n in np.arange(len(oo[0]) ):
    i,j=oo[0][n] , oo[1][n]
    plt.scatter( trx.lons[i,j]  , trx.lats[i,j] , color='white')

#itx=791
#plt.scatter( trx.lons[itx,:]  , trx.lats[itx,:] , color='white')

In [ ]:
print(oo[0][1])

In [ ]:
for n in np.arange(len(oo[0])):
    print(n)
    i,j=oo[0][n] , oo[1][n]
    print(i,j)

In [ ]:
tr=tracks[402]
loop=np.zeros(10)
print(lon[ np.array(tr['ix'][0:1])]  )
loop[0:1] = lon[ np.array(tr['ix'][0:1])] 